# Part 1: Code Q&A System with Bash Tools

This notebook implements a RAG system that answers questions about the `mcp-gateway-registry` codebase using bash tools for context retrieval.

## Architecture
```
User Query --> Query Router --> Bash Tool(s) --> LLM Answer
              (classify)       (Execute & Get    (with Context)
                                File Contents)
```

Instead of traditional vector-based retrieval, we use bash tools (grep, find, cat, tree) to retrieve relevant context from the codebase.

## Setup

In [11]:
import os
import subprocess
import logging
from pathlib import Path
from dotenv import load_dotenv
import litellm

# Load environment variables
load_dotenv(Path('../.env'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
MODEL = 'groq/llama-3.3-70b-versatile'
CODEBASE_PATH = Path('../mcp-gateway-registry').resolve()

print(f"Groq API Key loaded: {'Yes' if GROQ_API_KEY else 'No'}")
print(f"Codebase path: {CODEBASE_PATH}")
print(f"Codebase exists: {CODEBASE_PATH.exists()}")

Groq API Key loaded: Yes
Codebase path: /Users/akshayarun/Desktop/Spring_2026/DSAN-6725/spring-2026-a03-akshay17arun/mcp-gateway-registry
Codebase exists: True


## Query Classification

The query router classifies the incoming question to determine what type of information is needed and which bash tools to use.

In [12]:
def classify_query(question: str) -> str:
    """
    Classify the query type to determine which bash tools to use.

    Returns one of:
    - 'dependencies': Questions about packages/dependencies
    - 'structure': Questions about repo structure, entry points, file types
    - 'auth': Questions about authentication/authorization flows
    - 'api': Questions about API endpoints, routes
    - 'multi': Complex questions requiring multiple sources
    """
    q = question.lower()

    if any(kw in q for kw in ['depend', 'package', 'library', 'requirement', 'install', 'pyproject', 'package.json']):
        return 'dependencies'

    if any(kw in q for kw in ['entry point', 'main file', 'main entry', 'programming language', 'file type', 'directory', 'structure', 'organize']):
        return 'structure'

    if any(kw in q for kw in ['oauth', 'okta', 'provider', 'new auth']):
        return 'multi'

    if any(kw in q for kw in ['auth', 'token', 'login', 'permission', 'scope', 'jwt', 'credential', 'authoriz']):
        if any(kw in q for kw in ['endpoint', 'api', 'route']):
            return 'api'
        return 'auth'

    if any(kw in q for kw in ['endpoint', 'api', 'route', 'rest', 'http']):
        return 'api'

    return 'multi'


# Test classification on all 6 questions
test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?"
]

for i, q in enumerate(test_questions, 1):
    qtype = classify_query(q)
    print(f"Q{i} [{qtype:15s}]: {q[:65]}...")

Q1 [dependencies   ]: What Python dependencies does this project use?...
Q2 [structure      ]: What is the main entry point file for the registry service?...
Q3 [structure      ]: What programming languages and file types are used in this reposi...
Q4 [auth           ]: How does the authentication flow work, from token validation to u...
Q5 [api            ]: What are all the API endpoints available in the registry service ...
Q6 [multi          ]: How would you add support for a new OAuth provider (e.g., Okta) t...


## Bash Tool Selection and Execution

Based on the query type, we select and execute appropriate bash commands to retrieve relevant context from the codebase.

In [13]:
def run_bash(cmd: str, cwd: str = None, timeout: int = 30) -> str:
    """Execute a bash command and return its stdout output."""
    logger.info(f"CMD: {cmd[:80]}")
    try:
        result = subprocess.run(
            cmd, shell=True, capture_output=True, text=True,
            cwd=cwd or str(CODEBASE_PATH), timeout=timeout
        )
        output = result.stdout
        if result.returncode != 0 and result.stderr:
            output += f"\n[stderr]: {result.stderr[:300]}"
        return output
    except subprocess.TimeoutExpired:
        return "[Command timed out]"
    except Exception as e:
        return f"[Error: {e}]"


def truncate(text: str, max_chars: int = 4000) -> str:
    """Truncate text to avoid exceeding context limits."""
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n... [truncated, showing first {max_chars} of {len(text)} chars]"


def retrieve_context_dependencies() -> tuple:
    """Retrieve Python and Node.js dependency information."""
    parts = []
    cmds = []

    # Find all pyproject.toml and requirements files
    cmd = "find . -name 'pyproject.toml' -o -name 'requirements*.txt' | grep -v '.git' | grep -v node_modules | sort"
    files = run_bash(cmd)
    cmds.append(cmd)
    for f in files.strip().split('\n'):
        if f.strip():
            c = f"cat '{f.strip()}'"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} ===\n{out}")
            cmds.append(c)

    # Find package.json files (Node.js deps)
    cmd2 = "find . -name 'package.json' -not -path '*/node_modules/*' -not -path '*/.git/*' | sort"
    pkg_files = run_bash(cmd2)
    cmds.append(cmd2)
    for f in pkg_files.strip().split('\n')[:4]:
        if f.strip():
            c = f"cat '{f.strip()}' | head -60"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} (Node.js) ===\n{out}")
            cmds.append(c)

    return '\n\n'.join(parts), cmds


def retrieve_context_structure() -> tuple:
    """Retrieve repository structure, entry points, and file type info."""
    parts = []
    cmds = []

    # Directory tree
    cmd1 = "tree -L 3 --dirsfirst -I 'node_modules|__pycache__|.git|*.pyc|dist|build|.venv' 2>/dev/null || find . -maxdepth 3 -not -path '*/.git/*' -not -path '*/node_modules/*' | sort | head -100"
    out1 = run_bash(cmd1)
    parts.append(f"=== Directory Tree (depth 3) ===\n{out1}")
    cmds.append(cmd1)

    # File type distribution
    cmd2 = "find . -not -path '*/.git/*' -not -path '*/node_modules/*' -not -path '*/__pycache__/*' -not -path '*/.venv/*' -type f | sed 's/.*\\.//' | sort | uniq -c | sort -rn | head -20"
    out2 = run_bash(cmd2)
    parts.append(f"=== File Extension Counts ===\n{out2}")
    cmds.append(cmd2)

    # Find main/entry point files
    cmd3 = "find . -name 'main.py' -o -name '__main__.py' -o -name 'app.py' -o -name 'server.py' | grep -v __pycache__ | grep -v node_modules | grep -v .venv | head -15"
    entry_files = run_bash(cmd3)
    parts.append(f"=== Entry Point Files ===\n{entry_files}")
    cmds.append(cmd3)

    # Read those entry point files
    for f in entry_files.strip().split('\n')[:3]:
        if f.strip():
            c = f"head -50 '{f.strip()}'"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} (first 50 lines) ===\n{out}")
            cmds.append(c)

    # pyproject.toml for scripts
    cmd4 = "cat pyproject.toml 2>/dev/null"
    out4 = run_bash(cmd4)
    parts.append(f"=== pyproject.toml ===\n{out4}")
    cmds.append(cmd4)

    # README overview
    cmd5 = "head -80 README.md 2>/dev/null"
    out5 = run_bash(cmd5)
    parts.append(f"=== README.md (first 80 lines) ===\n{out5}")
    cmds.append(cmd5)

    return '\n\n'.join(parts), cmds


def retrieve_context_auth() -> tuple:
    """Retrieve authentication flow information."""
    parts = []
    cmds = []

    # Find auth-related Python files
    cmd1 = "find . -path '*auth*' -name '*.py' | grep -v __pycache__ | grep -v node_modules | grep -v .venv | grep -v test | sort"
    auth_files = run_bash(cmd1)
    parts.append(f"=== Auth Python Files ===\n{auth_files}")
    cmds.append(cmd1)

    for f in auth_files.strip().split('\n')[:6]:
        if f.strip():
            c = f"cat '{f.strip()}'"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} ===\n{truncate(out, 2500)}")
            cmds.append(c)

    # Search for token validation functions
    cmd2 = r"grep -rn 'def.*token\|def.*auth\|def.*verify\|def.*validate' --include='*.py' | grep -v __pycache__ | grep -v .venv | head -30"
    out2 = run_bash(cmd2)
    parts.append(f"=== Token/Auth Function Definitions ===\n{out2}")
    cmds.append(cmd2)

    # JWT and OAuth usage
    cmd3 = r"grep -rn 'jwt\|JWT\|Bearer\|oauth\|OAuth' --include='*.py' | grep -v __pycache__ | grep -v .venv | head -30"
    out3 = run_bash(cmd3)
    parts.append(f"=== JWT/OAuth Usage ===\n{out3}")
    cmds.append(cmd3)

    # Auth docs
    cmd4 = r"find ./docs -name '*.md' 2>/dev/null | xargs grep -l 'auth\|token\|oauth' 2>/dev/null | head -5"
    doc_files = run_bash(cmd4)
    cmds.append(cmd4)
    for f in doc_files.strip().split('\n')[:2]:
        if f.strip():
            c = f"cat '{f.strip()}'"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} ===\n{truncate(out, 2000)}")
            cmds.append(c)

    return '\n\n'.join(parts), cmds


def retrieve_context_api() -> tuple:
    """Retrieve API endpoint and scope information."""
    parts = []
    cmds = []

    # Find FastAPI router files
    cmd1 = r"grep -rn 'APIRouter\|FastAPI\|include_router' --include='*.py' -l | grep -v __pycache__ | grep -v .venv | sort"
    router_files = run_bash(cmd1)
    parts.append(f"=== FastAPI Router Files ===\n{router_files}")
    cmds.append(cmd1)

    for f in router_files.strip().split('\n')[:6]:
        if f.strip():
            c = f"cat '{f.strip()}'"
            out = run_bash(c)
            parts.append(f"=== {f.strip()} ===\n{truncate(out, 3000)}")
            cmds.append(c)

    # Route decorators
    cmd2 = r"grep -rn '@router\.\|@app\.get\|@app\.post\|@app\.put\|@app\.delete\|@app\.patch' --include='*.py' | grep -v __pycache__ | grep -v .venv | head -50"
    out2 = run_bash(cmd2)
    parts.append(f"=== Route Decorator Definitions ===\n{out2}")
    cmds.append(cmd2)

    # Scope requirements
    cmd3 = r"grep -rn 'scope\|Security\|Depends\|required_scope' --include='*.py' | grep -v __pycache__ | grep -v .venv | head -40"
    out3 = run_bash(cmd3)
    parts.append(f"=== Security Scopes/Dependencies ===\n{out3}")
    cmds.append(cmd3)

    return '\n\n'.join(parts), cmds


def retrieve_context_multi(question: str) -> tuple:
    """Comprehensive multi-source retrieval for complex questions."""
    parts = []
    cmds = []
    q_lower = question.lower()

    # Directory overview
    cmd1 = "tree -L 2 --dirsfirst -I 'node_modules|__pycache__|.git|.venv' 2>/dev/null"
    out1 = run_bash(cmd1)
    parts.append(f"=== Top-Level Structure ===\n{out1}")
    cmds.append(cmd1)

    # For OAuth provider questions
    if any(kw in q_lower for kw in ['oauth', 'okta', 'provider']):
        for kw in ['oauth', 'OAuth', 'provider', 'OAuthProvider']:
            cmd = f"grep -rn '{kw}' --include='*.py' --include='*.md' --include='*.ts' | grep -v __pycache__ | grep -v node_modules | grep -v .venv | head -25"
            out = run_bash(cmd)
            parts.append(f"=== Search '{kw}' ===\n{out}")
            cmds.append(cmd)

        # Find OAuth-related source files and read them
        cmd2 = r"find . -name '*.py' | xargs grep -l 'oauth\|OAuth' 2>/dev/null | grep -v __pycache__ | grep -v .venv | grep -v test | head -8"
        oauth_files = run_bash(cmd2)
        cmds.append(cmd2)
        for f in oauth_files.strip().split('\n')[:4]:
            if f.strip():
                c = f"cat '{f.strip()}'"
                out = run_bash(c)
                parts.append(f"=== {f.strip()} ===\n{truncate(out, 2500)}")
                cmds.append(c)

        # Auth server files
        cmd3 = "find ./auth_server -name '*.py' | grep -v __pycache__ | grep -v .venv | sort 2>/dev/null"
        auth_server_files = run_bash(cmd3)
        cmds.append(cmd3)
        parts.append(f"=== Auth Server Files ===\n{auth_server_files}")
        for f in auth_server_files.strip().split('\n')[:5]:
            if f.strip():
                c = f"cat '{f.strip()}'"
                out = run_bash(c)
                parts.append(f"=== {f.strip()} ===\n{truncate(out, 2000)}")
                cmds.append(c)

        # Configuration files
        cmd4 = r"find . -name '*.yaml' -o -name '*.yml' -o -name '*.json' | grep -v node_modules | grep -v .git | grep -i 'auth\|oauth\|config' | head -10"
        out4 = run_bash(cmd4)
        parts.append(f"=== Auth Config Files ===\n{out4}")
        cmds.append(cmd4)

        # Docs
        cmd5 = "find ./docs -name '*.md' 2>/dev/null | head -10"
        docs = run_bash(cmd5)
        cmds.append(cmd5)
        for f in docs.strip().split('\n')[:3]:
            if f.strip():
                c = f"cat '{f.strip()}'"
                out = run_bash(c)
                parts.append(f"=== {f.strip()} ===\n{truncate(out, 1500)}")
                cmds.append(c)
    else:
        # Generic comprehensive search
        for d in ['registry', 'auth_server', 'docs']:
            cmd = f"find ./{d} -name '*.py' -o -name '*.md' 2>/dev/null | grep -v __pycache__ | head -10"
            files = run_bash(cmd)
            cmds.append(cmd)
            for f in files.strip().split('\n')[:3]:
                if f.strip():
                    c = f"cat '{f.strip()}'"
                    out = run_bash(c)
                    parts.append(f"=== {f.strip()} ===\n{truncate(out, 2000)}")
                    cmds.append(c)

    return '\n\n'.join(parts), cmds


def retrieve_context(question: str, query_type: str) -> dict:
    """Dispatch to the right retrieval strategy based on query type."""
    if query_type == 'dependencies':
        ctx, cmds = retrieve_context_dependencies()
    elif query_type == 'structure':
        ctx, cmds = retrieve_context_structure()
    elif query_type == 'auth':
        ctx, cmds = retrieve_context_auth()
    elif query_type == 'api':
        ctx, cmds = retrieve_context_api()
    else:
        ctx, cmds = retrieve_context_multi(question)

    # Final truncation to keep within LLM context limits
    ctx = truncate(ctx, 14000)
    return {'context': ctx, 'commands_used': cmds}


print("Bash tool retrieval functions defined.")

Bash tool retrieval functions defined.


## LLM Answer Generation

Pass the retrieved context + question to the LLM for a structured answer with file references.

In [14]:
def generate_answer(question: str, context: str) -> str:
    """Generate an answer using the LLM with the retrieved context."""

    system_prompt = """You are an expert code assistant that answers questions about software codebases.
You are given context retrieved from the codebase using bash tools (grep, find, cat, tree).

Your answers should:
- Directly answer the question with specific details
- Reference specific files, functions, and line numbers
- Use clear formatting with sections/bullets for complex answers
- Synthesize information from multiple files when needed
- Acknowledge uncertainty if the context is insufficient"""

    user_prompt = f"""Question: {question}

Context retrieved from the codebase using bash tools:
---
{context}
---

Please provide a detailed answer based on the above context, with specific file references."""

    response = litellm.completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=2000
    )

    return response.choices[0].message.content


def answer_question(question: str) -> dict:
    """Full pipeline: classify -> retrieve -> generate."""
    print(f"\n{'='*70}")
    print(f"Question: {question}")
    print('='*70)

    query_type = classify_query(question)
    print(f"Query type: {query_type}")

    print("Retrieving context via bash tools...")
    retrieval = retrieve_context(question, query_type)
    context = retrieval['context']
    commands_used = retrieval['commands_used']
    print(f"Executed {len(commands_used)} commands | Context: {len(context)} chars")

    print("Generating answer with LLM...")
    answer = generate_answer(question, context)

    print(f"\nAnswer:\n{answer}")

    return {
        'question': question,
        'query_type': query_type,
        'commands_used': commands_used,
        'context_length': len(context),
        'answer': answer
    }


print("LLM generation functions defined.")

LLM generation functions defined.


## Answer All Test Questions

In [17]:
import time

test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?"
]

results = []
for i, question in enumerate(test_questions, 1):
    print(f"\n\n{'#'*70}")
    print(f"# Question {i}/{len(test_questions)}")
    print('#'*70)
    result = answer_question(question)
    results.append(result)
    if i < len(test_questions):
        time.sleep(60)

print("\n\nAll 6 questions answered!")

2026-03-05 19:57:06,035 - INFO - CMD: find . -name 'pyproject.toml' -o -name 'requirements*.txt' | grep -v '.git' | gr
2026-03-05 19:57:06,071 - INFO - CMD: cat './agents/a2a/pyproject.toml'
2026-03-05 19:57:06,094 - INFO - CMD: cat './auth_server/pyproject.toml'
2026-03-05 19:57:06,107 - INFO - CMD: cat './metrics-service/pyproject.toml'
2026-03-05 19:57:06,123 - INFO - CMD: cat './pyproject.toml'
2026-03-05 19:57:06,137 - INFO - CMD: cat './servers/currenttime/pyproject.toml'
2026-03-05 19:57:06,152 - INFO - CMD: cat './servers/example-server/pyproject.toml'
2026-03-05 19:57:06,166 - INFO - CMD: cat './servers/fininfo/pyproject.toml'
2026-03-05 19:57:06,184 - INFO - CMD: cat './servers/mcpgw/pyproject.toml'
2026-03-05 19:57:06,199 - INFO - CMD: cat './servers/realserverfaketools/pyproject.toml'
2026-03-05 19:57:06,212 - INFO - CMD: cat './terraform/aws-ecs/scripts/requirements.txt'
2026-03-05 19:57:06,224 - INFO - CMD: find . -name 'package.json' -not -path '*/node_modules/*' -not -p



######################################################################
# Question 1/6
######################################################################

Question: What Python dependencies does this project use?
Query type: dependencies
Retrieving context via bash tools...


2026-03-05 19:57:06,250 - INFO - CMD: cat './cli/package.json' | head -60
2026-03-05 19:57:06,264 - INFO - CMD: cat './frontend/package.json' | head -60
2026-03-05 19:57:06,278 - INFO - CMD: cat './package.json' | head -60
19:57:06 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
2026-03-05 19:57:06,295 - INFO - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq


Executed 15 commands | Context: 13287 chars
Generating answer with LLM...


19:57:11 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
2026-03-05 19:57:11,328 - INFO - Wrapper: Completed Call, calling success_handler


...



######################################################################
# Question 6/6
######################################################################

Question: How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?
Query type: multi
Retrieving context via bash tools...


2026-03-05 20:02:22,148 - INFO - CMD: grep -rn 'provider' --include='*.py' --include='*.md' --include='*.ts' | grep -v
2026-03-05 20:02:22,202 - INFO - CMD: grep -rn 'OAuthProvider' --include='*.py' --include='*.md' --include='*.ts' | gr
2026-03-05 20:02:22,263 - INFO - CMD: find . -name '*.py' | xargs grep -l 'oauth\|OAuth' 2>/dev/null | grep -v __pycac
2026-03-05 20:02:22,367 - INFO - CMD: cat './auth_server/server.py'
2026-03-05 20:02:22,376 - INFO - CMD: cat './auth_server/providers/cognito.py'
2026-03-05 20:02:22,384 - INFO - CMD: cat './auth_server/providers/entra.py'
2026-03-05 20:02:22,392 - INFO - CMD: cat './auth_server/providers/keycloak.py'
2026-03-05 20:02:22,400 - INFO - CMD: find ./auth_server -name '*.py' | grep -v __pycache__ | grep -v .venv | sort 2>/
2026-03-05 20:02:22,410 - INFO - CMD: cat './auth_server/__init__.py'
2026-03-05 20:02:22,419 - INFO - CMD: cat './auth_server/cognito_utils.py'
2026-03-05 20:02:22,427 - INFO - CMD: cat './auth_server/metrics_middleware

Executed 21 commands | Context: 14052 chars
Generating answer with LLM...


20:02:25 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
2026-03-05 20:02:25,468 - INFO - Wrapper: Completed Call, calling success_handler



Answer:
Adding Support for a New OAuth Provider

To add support for a new OAuth provider, such as Okta, you will need to modify the following files:

### 1. `auth_server/oauth2_providers.yml`

This file contains the configuration for all OAuth providers. You will need to add a new entry for Okta, including its client ID, client secret, and authorization URL.

Example:
```yml
okta:
  client_id: YOUR_OKTA_CLIENT_ID
  client_secret: YOUR_OKTA_CLIENT_SECRET
  authorization_url: https://your-okta-domain.com/oauth2/v1/authorize
  token_url: https://your-okta-domain.com/oauth2/v1/token
```
### 2. `auth_server/server.py`

This file contains the logic for handling OAuth authentication. You will need to add a new function to handle the Okta OAuth flow.

Example:
```python
@app.get("/oauth2/login/okta")
async def okta_login(request: Request, redirect_uri: str = None):
    # Redirect the user to the Okta authorization URL
    auth_url = f"{okta_config['authorization_url']}?client_id={okta_config[

## Save Results to part1_results.txt

In [18]:
output_path = Path('../part1_results.txt')

with open(output_path, 'w') as f:
    f.write("# Part 1: Code Q&A System with Bash Tools\n")
    f.write("# mcp-gateway-registry codebase\n")
    f.write("=" * 70 + "\n\n")

    for i, result in enumerate(results, 1):
        f.write(f"{'='*70}\n")
        f.write(f"Question {i}: {result['question']}\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"Query Type: {result['query_type']}\n\n")
        f.write(f"Bash Commands Used ({len(result['commands_used'])}):\n")
        for cmd in result['commands_used']:
            f.write(f"  $ {cmd[:120]}\n")
        f.write(f"\nContext Retrieved: {result['context_length']} characters\n\n")
        f.write(f"Answer:\n{result['answer']}\n\n")

print(f"Results saved to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Results saved to /Users/akshayarun/Desktop/Spring_2026/DSAN-6725/spring-2026-a03-akshay17arun/part1_results.txt
File size: 28,039 bytes
